# Run `28aee20060a4`

**Mode**: `download`
**Tier**: `deep`
**Datasets**: _none selected_

## Request

> Finish downloading and configuring GSE232200 and GSE136802.

## Final summary

All work is complete. Both GEO accessions are downloaded, configured, and committed to the lab registry.

## Summary
I finished acquiring and configuring both datasets from NCBI GEO. **GSE232200** (Green et al. 2023, mouse PPC inhibitory-neuron snATAC-seq) is now a merged peak×cell matrix of **22,988 cells × 459,912 mm10 peaks**, built from the two bioreps' SnapATAC `.snap` files. **GSE136802** (Hrvatin et al. 2019 PESCA interneuron data) is two artifacts: a **47,348-cell × 27,035-gene** scRNA matrix and a **pseudobulk ATAC** matrix (6 sorted Sst/Vip/Pv samples × 323,369 peaks). Both are registered and live for all future jobs.

## What I did
- Enumerated GEO file listings; staged all 8 public files (4.31 GB).
- Built and verified `gse232200_snapatac.h5ad` (peak×cell, with per-cell QC), `gse136802_scrna.h5ad` (raw + lognorm layers), `gse136802_atac.h5ad` (counts + binary layer, GRE annotations).
- Wrote `gse232200_config.md` / `gse136802_config.md` provenance notes.
- Registered `green-2023-ppc-inhibitory-snatac` (6 files, 6.1 GB) and `hrvatin-2019-pesca-interneuron` (7 files, 0.59 GB), each with a dataset card. Saved a lab note.

## Caveats / next steps
- **63 orphan peaks** (idy 459913–459975) in the `.snap` PM group had no stored coordinates (~0.008% of nonzeros) and were dropped; the coordinate array is a valid genomic-sorted prefix.
- GSE136802 scRNA has **no deposited Sst/Vip/Pv labels** — assign by clustering. Its ATAC is **pseudobulk**, not single-cell.
- Not extracted from the raw `.snap` (still available): the 5kb `AM/5000` bin×cell matrix and `FM` raw fragments — easy to add if needed.

A lighter tier could have handled the downloads, but the 8 GB-RAM matrix assembly and `.snap` schema reverse-engineering justified deep tier.

In [ ]:
import os, glob
for base in ['/work']:
    for root, dirs, files in os.walk(base):
        # skip deep hidden
        if '.ipython' in root or '.cache' in root: 
            continue
        depth = root[len(base):].count(os.sep)
        if depth > 3: 
            continue
        for f in files:
            p = os.path.join(root,f)
            try:
                sz = os.path.getsize(p)
            except: 
                sz = -1
            print(f"{sz/1e6:10.2f} MB  {p}")


exit_code: 0
--- stdout ---
      0.00 MB  /work/.kernel.json
      0.00 MB  /work/_cell_0001.py
      0.02 MB  /work/.mpl_cache/fontlist-v390.json

--- stderr ---



In [ ]:
import urllib.request, ssl, json
ctx = ssl.create_default_context()

def listing(url):
    try:
        req = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
        return urllib.request.urlopen(req, timeout=60, context=ctx).read().decode('utf-8', 'replace')
    except Exception as e:
        return f"ERR {e}"

# GEO series suppl listings via HTTPS FTP mirror
for acc, nnn in [('GSE232200','GSE232nnn'),('GSE136802','GSE136nnn')]:
    url = f"https://ftp.ncbi.nlm.nih.gov/geo/series/{nnn}/{acc}/suppl/"
    print("="*70)
    print(url)
    print(listing(url)[:3000])


exit_code: 0
--- stdout ---
https://ftp.ncbi.nlm.nih.gov/geo/series/GSE232nnn/GSE232200/suppl/
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 3.2 Final//EN">
<html>
 <head>
  <title>Index of /geo/series/GSE232nnn/GSE232200/suppl</title>
 </head>
 <body>
<h1>Index of /geo/series/GSE232nnn/GSE232200/suppl</h1>
<pre>Name                             Last modified      Size  <hr><a href="/geo/series/GSE232nnn/GSE232200/">Parent Directory</a>                                      -   
<a href="GSE232200_RAW.tar">GSE232200_RAW.tar</a>                2023-05-12 10:15  4.3G  
<a href="filelist.txt">filelist.txt</a>                     2023-05-12 10:15  379   
<hr></pre>
<a href="https://www.hhs.gov/vulnerability-disclosure-policy/index.html">HHS Vulnerability Disclosure</a>

https://ftp.ncbi.nlm.nih.gov/geo/series/GSE136nnn/GSE136802/suppl/
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 3.2 Final//EN">
<html>
 <head>
  <title>Index of /geo/series/GSE136nnn/GSE136802/suppl</title>
 </head>
 <body>
<h1>Index of

In [ ]:
for acc,nnn in [('GSE232200','GSE232nnn'),('GSE136802','GSE136nnn')]:
    url=f"https://ftp.ncbi.nlm.nih.gov/geo/series/{nnn}/{acc}/suppl/filelist.txt"
    print("="*70); print(url)
    print(listing(url))

# per-sample for GSE232200 bioreps
for gsm,nnn in [('GSM7317753','GSM7317nnn'),('GSM7317754','GSM7317nnn')]:
    url=f"https://ftp.ncbi.nlm.nih.gov/geo/samples/{nnn}/{gsm}/suppl/"
    print("="*70); print(url)
    print(listing(url)[:2000])


exit_code: 0
--- stdout ---
https://ftp.ncbi.nlm.nih.gov/geo/series/GSE232nnn/GSE232200/suppl/filelist.txt
#Archive/File	Name	Time	Size	Type
Archive	GSE232200_RAW.tar	05/12/2023 10:15:31	4566507520	TAR
File	GSM7317753_biorep1.snap.gz	05/10/2023 14:35:20	2258461416	SNAP
File	GSM7317753_biorep1_singlecell.csv.gz	05/10/2023 14:35:23	5498948	CSV
File	GSM7317754_biorep2.snap.gz	05/10/2023 14:38:29	2296962551	SNAP
File	GSM7317754_biorep2_singlecell.csv.gz	05/10/2023 14:38:32	5579497	CSV

https://ftp.ncbi.nlm.nih.gov/geo/series/GSE136nnn/GSE136802/suppl/filelist.txt
#Archive/File	Name	Time	Size	Type
Archive	GSE136802_RAW.tar	10/16/2019 08:31:04	51486720	TAR
File	GSM4058339_180320_6-19-1-1_WITH_BARCODES.counts.tsv.gz	09/03/2019 15:08:51	5098135	TSV
File	GSM4058339_180320_6-19-1-2_WITH_BARCODES.counts.tsv.gz	09/03/2019 15:08:54	4833078	TSV
File	GSM4058339_180320_6-19-1-3_WITH_BARCODES.counts.tsv.gz	09/03/2019 15:08:58	4961280	TSV
File	GSM4058339_180320_6-19-1-4_WITH_BARCODES.counts.tsv.gz	09/03

In [ ]:
import tarfile, os, gzip, pandas as pd, glob
base='/work/staged/hrvatin-2019-pesca-interneuron'
print(os.listdir(base))
# peek inside RAW.tar
with tarfile.open(os.path.join(base,'GSE136802_RAW.tar')) as t:
    names=t.getnames()
print(f"\nRAW.tar has {len(names)} members")
for n in names[:30]: print('  ',n)
print('  ...' if len(names)>30 else '')
# categorize
import collections
def cat(n):
    if n.endswith('.counts.tsv.gz'): return 'scRNA counts tsv'
    if n.endswith('.bed.gz'): return 'atac bed'
    return 'other'
c=collections.Counter(cat(n) for n in names)
print("\ncategories:",dict(c))


exit_code: 0
--- stdout ---
['GSE136802_atac_master_counts.csv.gz', 'GSE136802_RAW.tar', 'GSE136802_S2_annotated_gres.csv.gz', 'GSE136802_atac_master_binary_peak_table.csv.gz']

RAW.tar has 16 members
   GSM4058339_180320_6-19-1-1_WITH_BARCODES.counts.tsv.gz
   GSM4058339_180320_6-19-1-2_WITH_BARCODES.counts.tsv.gz
   GSM4058339_180320_6-19-1-3_WITH_BARCODES.counts.tsv.gz
   GSM4058339_180320_6-19-1-4_WITH_BARCODES.counts.tsv.gz
   GSM4058339_180320_6-19-1-5_WITH_BARCODES.counts.tsv.gz
   GSM4058339_180320_6-19-2-1_WITH_BARCODES.counts.tsv.gz
   GSM4058339_180320_6-19-2-2_WITH_BARCODES.counts.tsv.gz
   GSM4058339_180320_6-19-2-3_WITH_BARCODES.counts.tsv.gz
   GSM4058339_180320_6-19-2-4_WITH_BARCODES.counts.tsv.gz
   GSM4058339_180320_6-19-2-5_WITH_BARCODES.counts.tsv.gz
   GSM4058340_macs_homer_intersect_CTX_PV_1.bed.gz
   GSM4058341_macs_homer_intersect_CTX_PV_2.bed.gz
   GSM4058342_macs_homer_intersect_CTX_VIP_1.bed.gz
   GSM4058343_macs_homer_intersect_CTX_VIP_2.bed.gz
   GSM4058344

In [ ]:
import tarfile, gzip, io, pandas as pd
base='/work/staged/hrvatin-2019-pesca-interneuron'
t=tarfile.open(os.path.join(base,'GSE136802_RAW.tar'))

# peek one scRNA counts tsv
m='GSM4058339_180320_6-19-1-1_WITH_BARCODES.counts.tsv.gz'
f=t.extractfile(m)
raw=gzip.decompress(f.read())
head=raw[:800].decode('utf-8','replace')
print("=== scRNA counts tsv head ===")
print(head)
print("\n... lines:", raw.count(b'\n'))

# atac master counts
print("\n=== atac_master_counts.csv head ===")
with gzip.open(os.path.join(base,'GSE136802_atac_master_counts.csv.gz'),'rt') as fh:
    for i,line in enumerate(fh):
        print(line.rstrip()[:300])
        if i>4: break

print("\n=== S2_annotated_gres.csv head ===")
with gzip.open(os.path.join(base,'GSE136802_S2_annotated_gres.csv.gz'),'rt') as fh:
    for i,line in enumerate(fh):
        print(line.rstrip()[:300])
        if i>4: break


exit_code: 0
--- stdout ---
=== scRNA counts tsv head ===
	0610005C13Rik	0610006L08Rik	0610007P14Rik	0610009B22Rik	0610009E02Rik	0610009L18Rik	0610009O20Rik	0610010F05Rik	0610010K14Rik	0610012D04Rik	0610012G03Rik	0610025J13Rik	0610030E20Rik	0610031O16Rik	0610033M10Rik	0610037L13Rik	0610038B21Rik	0610040B10Rik	0610040F04Rik	0610040J01Rik	0610043K17Rik	1010001B22Rik	1010001N08Rik	1110002J07Rik	1110002L01Rik	1110002O04Rik	1110004E09Rik	1110004F10Rik	1110008F13Rik	1110008L16Rik	1110008P14Rik	1110012L19Rik	1110017D15Rik	1110018N20Rik	1110019D14Rik	1110020A21Rik	1110025L11Rik	1110028F11Rik	1110028F18Rik	1110032A03Rik	1110032F04Rik	1110034G24Rik	1110036E04Rik	1110037F02Rik	1110038B12Rik	1110038F14Rik	1110046J04Rik	1110051M20Rik	1110057P08Rik	1110059E24Rik	1110059G10Rik	1110065P20Rik	1190002N15Rik	1190003K10Rik	1190005I06Rik	1190007I07Rik	1190028D05Rik	1

... lines: 5124

=== atac_master_counts.csv head ===
,CTX_PV_1,CTX_PV_2,CTX_VIP_1,CTX_VIP_2,CTX_SST_1,CTX_SST_2
master_1,0.0,1.0,0.0,1.0,0.0

In [ ]:
import scanpy as sc, anndata as ad, numpy as np, scipy.sparse as sp, pandas as pd, gzip, io, tarfile, os
base='/work/staged/hrvatin-2019-pesca-interneuron'
t=tarfile.open(os.path.join(base,'GSE136802_RAW.tar'))
rna_members=[n for n in t.getnames() if n.endswith('.counts.tsv.gz')]

adatas=[]
for m in sorted(rna_members):
    raw=gzip.decompress(t.extractfile(m).read())
    df=pd.read_csv(io.BytesIO(raw), sep='\t', index_col=0)
    lane=m.split('_WITH')[0].replace('GSM4058339_','')
    df.index=[f"{lane}|{bc}" for bc in df.index]
    a=ad.AnnData(sp.csr_matrix(df.values.astype(np.float32)),
                 obs=pd.DataFrame({'lane':lane}, index=df.index),
                 var=pd.DataFrame(index=df.columns.astype(str)))
    adatas.append(a)
    print(m.split('_WITH')[0], df.shape)

rna = ad.concat(adatas, join='outer', fill_value=0)
rna.obs['gsm']='GSM4058339'; rna.obs['assay']='scRNA-seq (INTACT cortical interneurons)'
print("\nMerged scRNA:", rna.shape, "genes:", rna.n_vars)
print("counts dtype, max:", rna.X.dtype, rna.X.max())
rna.write('/work/gse136802_scrna.h5ad')
print("saved gse136802_scrna.h5ad")


exit_code: 0
--- stdout ---
GSM4058339_180320_6-19-1-1 (5123, 27035)
GSM4058339_180320_6-19-1-2 (5212, 27035)
GSM4058339_180320_6-19-1-3 (5072, 27035)
GSM4058339_180320_6-19-1-4 (5421, 27035)
GSM4058339_180320_6-19-1-5 (4478, 27035)
GSM4058339_180320_6-19-2-1 (4431, 27035)
GSM4058339_180320_6-19-2-2 (4328, 27035)
GSM4058339_180320_6-19-2-3 (4337, 27035)
GSM4058339_180320_6-19-2-4 (5042, 27035)
GSM4058339_180320_6-19-2-5 (3904, 27035)

Merged scRNA: (47348, 27035) genes: 27035
counts dtype, max: float32 1202.0
saved gse136802_scrna.h5ad

--- stderr ---



In [ ]:
import numpy as np, scipy.sparse as sp
# --- normalized layer for scRNA (raw counts kept in X and layers['counts']) ---
rna.layers['counts']=rna.X.copy()
counts=np.asarray(rna.X.sum(1)).ravel(); counts[counts==0]=1
norm=sp.diags(1e4/counts).dot(rna.X).tocsr()
norm.data=np.log1p(norm.data)
rna.layers['lognorm']=norm
rna.uns['normalization']='layers[counts]=raw; layers[lognorm]=log1p(CP10k); X=raw counts'
rna.write('/work/gse136802_scrna.h5ad')
print("scRNA re-saved with counts+lognorm layers", rna.shape)

# --- ATAC pseudobulk h5ad ---
import pandas as pd, gzip, anndata as ad
base='/work/staged/hrvatin-2019-pesca-interneuron'
cnt=pd.read_csv(gzip.open(base+'/GSE136802_atac_master_counts.csv.gz','rt'), index_col=0)
binr=pd.read_csv(gzip.open(base+'/GSE136802_atac_master_binary_peak_table.csv.gz','rt'), index_col=0)
gres=pd.read_csv(gzip.open(base+'/GSE136802_S2_annotated_gres.csv.gz','rt'), index_col=0)
print("counts:",cnt.shape,"binary:",binr.shape,"gres:",gres.shape)
print("counts cols:",list(cnt.columns))
print("binary aligns:", binr.shape==cnt.shape and (binr.index==cnt.index).all())

# AnnData: obs = 6 pseudobulk samples, var = peaks
X=cnt.T.values.astype(np.float32)            # 6 samples x Npeaks
obs=pd.DataFrame(index=cnt.columns)
obs['celltype']=[c.split('_')[1] for c in obs.index]
obs['replicate']=[c.split('_')[2] for c in obs.index]
obs['region']=[c.split('_')[0] for c in obs.index]
var=pd.DataFrame(index=cnt.index)
# attach GRE annotations to the subset of peaks that are annotated enhancers
for col in ['Chr','Start','End','Annotation','Gene Name','Name','ATAC_Specificity','PESCA_Specificity']:
    var[col.replace(' ','_')]=gres[col].reindex(var.index)
var['is_annotated_GRE']=var.index.isin(gres.index)
atac=ad.AnnData(sp.csr_matrix(X), obs=obs, var=var)
atac.layers['binary']=sp.csr_matrix(binr.T.reindex(index=cnt.columns)[cnt.index].values.astype(np.float32))
atac.uns['description']='Pseudobulk ATAC master peak x sample count matrix (PESCA, Hrvatin 2019). X=fragment counts; layers[binary]=binary peak calls.'
atac.write('/work/gse136802_atac.h5ad')
print("\nATAC saved:",atac.shape,"| GREs annotated:",int(atac.var['is_annotated_GRE'].sum()))
print(atac.obs)


exit_code: 0
--- stdout ---
scRNA re-saved with counts+lognorm layers (47348, 27035)
counts: (323369, 6) binary: (323369, 12) gres: (287, 13)
counts cols: ['CTX_PV_1', 'CTX_PV_2', 'CTX_VIP_1', 'CTX_VIP_2', 'CTX_SST_1', 'CTX_SST_2']
binary aligns: False

ATAC saved: (6, 323369) | GREs annotated: 287
          celltype replicate region
CTX_PV_1        PV         1    CTX
CTX_PV_2        PV         2    CTX
CTX_VIP_1      VIP         1    CTX
CTX_VIP_2      VIP         2    CTX
CTX_SST_1      SST         1    CTX
CTX_SST_2      SST         2    CTX

--- stderr ---



In [ ]:
print("binary columns:", list(binr.columns))
import numpy as np
lay=atac.layers['binary'].toarray()
print("binary layer NaN frac:", np.isnan(lay).mean())


exit_code: 0
--- stdout ---
binary columns: ['Chr', 'Start', 'End', 'CTX_PV_1', 'CTX_PV_2', 'CTX_VIP_1', 'CTX_VIP_2', 'CTX_SST_1', 'CTX_SST_2', 'MeanCons', 'MinCons', 'MaxCons']
binary layer NaN frac: 0.0

--- stderr ---



In [ ]:
# add full-peak coordinates + conservation from binary table to var
for col in ['Chr','Start','End','MeanCons','MinCons','MaxCons']:
    atac.var[col]=binr[col].reindex(atac.var.index).values
# prefer full coords; keep GRE-specific annotation columns too
atac.var=atac.var.rename(columns={'Chr':'chr','Start':'start','End':'end'})
# drop the now-redundant GRE coord cols (Chr_/Start_/End_ from gres were same name->already 'Chr'? they were 'Chr','Start','End' -> overwritten). 
print(atac.var.columns.tolist())
print(atac.var.head(3))
print("\npeaks with coords:", atac.var['chr'].notna().sum(),"/",atac.n_vars)
atac.write('/work/gse136802_atac.h5ad')
print("ATAC re-saved")


exit_code: 0
--- stdout ---
['chr', 'start', 'end', 'Annotation', 'Gene_Name', 'Name', 'ATAC_Specificity', 'PESCA_Specificity', 'is_annotated_GRE', 'MeanCons', 'MinCons', 'MaxCons']
           chr    start      end Annotation Gene_Name Name  ATAC_Specificity  \
master_1  chr1  3020259  3021015        NaN       NaN  NaN               NaN   
master_2  chr1  3037525  3038126        NaN       NaN  NaN               NaN   
master_3  chr1  3052781  3053382        NaN       NaN  NaN               NaN   

          PESCA_Specificity  is_annotated_GRE  MeanCons  MinCons  MaxCons  
master_1                NaN             False    0.0000    0.000    0.000  
master_2                NaN             False    0.0165   -1.479    0.278  
master_3                NaN             False    0.0000    0.000    0.000  

peaks with coords: 323369 / 323369
ATAC re-saved

--- stderr ---



In [ ]:
import subprocess, os, gzip, shutil, h5py
base='/work/staged/green-2023-ppc-inhibitory-snatac'
# decompress snap.gz -> .snap (need uncompressed to open with h5py)
for gsm in ['GSM7317753_biorep1','GSM7317754_biorep2']:
    src=f"{base}/{gsm}.snap.gz"; dst=f"/work/{gsm}.snap"
    if not os.path.exists(dst):
        with gzip.open(src,'rb') as fi, open(dst,'wb') as fo:
            shutil.copyfileobj(fi, fo, length=64*1024*1024)
    print(dst, f"{os.path.getsize(dst)/1e9:.2f} GB")

# inspect HDF5 structure of biorep1
f=h5py.File('/work/GSM7317753_biorep1.snap','r')
def show(name,obj):
    import h5py as _h
    if isinstance(obj,_h.Dataset):
        print(f"  D {name:40s} shape={obj.shape} dtype={obj.dtype}")
    else:
        print(f"  G {name}")
f.visititems(show)
print("\nroot attrs:", dict(f.attrs))


exit_code: 0
--- stdout ---
/work/GSM7317753_biorep1.snap 5.48 GB
/work/GSM7317754_biorep2.snap 5.57 GB
  G AM
  G AM/5000
  D AM/5000/binChrom                         shape=(546206,) dtype=object
  D AM/5000/binStart                         shape=(546206,) dtype=uint32
  D AM/5000/count                            shape=(141942181,) dtype=uint8
  D AM/5000/idx                              shape=(141942181,) dtype=uint32
  D AM/5000/idy                              shape=(141942181,) dtype=uint32
  D AM/binSizeList                           shape=(1,) dtype=uint32
  D AM/nBinSize                              shape=() dtype=uint32
  G BD
  D BD/CM                                    shape=(10614,) dtype=uint32
  D BD/PE                                    shape=(10614,) dtype=uint32
  D BD/PL                                    shape=(10614,) dtype=uint32
  D BD/PP                                    shape=(10614,) dtype=uint32
  D BD/SA                                    shape=(10614,) dtyp

In [ ]:
import gzip, pandas as pd, h5py, numpy as np
base='/work/staged/green-2023-ppc-inhibitory-snatac'
# singlecell.csv
sc1=pd.read_csv(gzip.open(base+'/GSM7317753_biorep1_singlecell.csv.gz','rt'))
print("singlecell.csv shape:",sc1.shape,"\ncols:",list(sc1.columns)[:25])
print(sc1.head(3).to_string())

f1=h5py.File('/work/GSM7317753_biorep1.snap','r')
f2=h5py.File('/work/GSM7317754_biorep2.snap','r')
# peak coordinate check
def peaks(f):
    return (np.array(f['PM/peakChrom']).astype(str), np.array(f['PM/peakStart']), np.array(f['PM/peakEnd']))
c1,s1,e1=peaks(f1); c2,s2,e2=peaks(f2)
print("\nbiorep1 npeaks:",len(c1),"biorep2 npeaks:",len(c2))
print("peak sets identical:", len(c1)==len(c2) and (c1==c2).all() and (s1==s2).all() and (e1==e2).all())
print("idx range b1:", f1['PM/idx'][:].min() if False else 'skip')
print("first peaks:", list(zip(c1[:3],s1[:3],e1[:3])))


exit_code: 0
--- stdout ---
singlecell.csv shape: (541151, 18) 
cols: ['barcode', 'total', 'duplicate', 'chimeric', 'unmapped', 'lowmapq', 'mitochondrial', 'passed_filters', 'cell_id', 'is__cell_barcode', 'TSS_fragments', 'DNase_sensitive_region_fragments', 'enhancer_region_fragments', 'promoter_region_fragments', 'on_target_fragments', 'blacklist_region_fragments', 'peak_region_fragments', 'peak_region_cutsites']
              barcode     total  duplicate  chimeric  unmapped  lowmapq  mitochondrial  passed_filters  cell_id  is__cell_barcode  TSS_fragments  DNase_sensitive_region_fragments  enhancer_region_fragments  promoter_region_fragments  on_target_fragments  blacklist_region_fragments  peak_region_fragments  peak_region_cutsites
0          NO_BARCODE  12011055    1705061     22984   3670031   471297          26552         6115130      NaN                 0              0                                 0                          0                          0                    0  

In [ ]:
import numpy as np, scipy.sparse as sp, pandas as pd, anndata as ad, h5py, gzip

def decode(arr): return np.array([x.decode() if isinstance(x,bytes) else str(x) for x in arr])

def build_biorep(f, label, sc_csv):
    bd_name=decode(np.array(f['BD/name']))
    nbar=len(bd_name)
    idx=f['PM/idx'][:].astype(np.int64)-1   # cell (0-based)
    idy=f['PM/idy'][:].astype(np.int64)-1   # peak (0-based)
    cnt=f['PM/count'][:]
    npeak=f['PM/peakStart'].shape[0]
    assert idx.max()<nbar and idy.max()<npeak, (idx.max(),nbar,idy.max(),npeak)
    X=sp.coo_matrix((cnt,(idx,idy)), shape=(nbar,npeak)).tocsr()
    # obs from BD QC fields
    obs=pd.DataFrame(index=[f"{label}|{b}" for b in bd_name])
    for k in ['TN','UM','PP','UQ','SE','SA','PE','PL','US','CM']:
        if f'BD/{k}' in f: obs[k]=f[f'BD/{k}'][:]
    obs['barcode']=bd_name; obs['biorep']=label
    # join singlecell.csv QC (on barcode)
    sc=sc_csv.set_index('barcode')
    keep=['passed_filters','peak_region_fragments','TSS_fragments','promoter_region_fragments',
          'enhancer_region_fragments','is__cell_barcode','cell_id']
    j=sc.reindex(bd_name)[keep]; j.index=obs.index
    obs=pd.concat([obs,j],axis=1)
    return X,obs

f1=h5py.File('/work/GSM7317753_biorep1.snap','r')
f2=h5py.File('/work/GSM7317754_biorep2.snap','r')
sc1=pd.read_csv(gzip.open(base+'/GSM7317753_biorep1_singlecell.csv.gz','rt'))
sc2=pd.read_csv(gzip.open(base+'/GSM7317754_biorep2_singlecell.csv.gz','rt'))

X1,o1=build_biorep(f1,'biorep1',sc1)
X2,o2=build_biorep(f2,'biorep2',sc2)
print("biorep1:",X1.shape,"nnz",X1.nnz,"| biorep2:",X2.shape,"nnz",X2.nnz)

# var from biorep1 peaks (identical across reps)
chrom=decode(np.array(f1['PM/peakChrom'])); st=f1['PM/peakStart'][:]; en=f1['PM/peakEnd'][:]
var=pd.DataFrame({'chr':chrom,'start':st,'end':en})
var.index=[f"{c}:{s}-{e}" for c,s,e in zip(chrom,st,en)]

X=sp.vstack([X1,X2]).tocsr()
obs=pd.concat([o1,o2])
atac=ad.AnnData(X.astype(np.float32), obs=obs, var=var)
atac.uns['description']='Green et al 2023 (GSE232200) snATAC-seq of cortical (PPC) Gad2+ inhibitory neurons; peak x cell matrix from SnapATAC .snap PM group, 2 bioreps merged.'
atac.uns['gsm']={'biorep1':'GSM7317753','biorep2':'GSM7317754'}
print("\nMerged:",atac.shape,"nnz",atac.X.nnz)
print(atac.obs[['biorep','passed_filters','peak_region_fragments','UQ']].groupby(atac.obs['biorep']).median())
atac.write('/work/gse232200_snapatac.h5ad')
print("saved /work/gse232200_snapatac.h5ad")


exit_code: 1
--- stdout ---

--- stderr ---

AssertionError: (10613, 10614, 459974, 459912)
---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
Cell In[1], line 32
     28 f2=h5py.File('/work/GSM7317754_biorep2.snap','r')
     29 sc1=pd.read_csv(gzip.open(base+'/GSM7317753_biorep1_singlecell.csv.gz','rt'))
     30 sc2=pd.read_csv(gzip.open(base+'/GSM7317754_biorep2_singlecell.csv.gz','rt'))
     31 
---> 32 X1,o1=build_biorep(f1,'biorep1',sc1)
     33 X2,o2=build_biorep(f2,'biorep2',sc2)
     34 print("biorep1:",X1.shape,"nnz",X1.nnz,"| biorep2:",X2.shape,"nnz",X2.nnz)
     35 

Cell In[1], line 12, in build_biorep(f, label, sc_csv)
      8     idx=f['PM/idx'][:].astype(np.int64)-1   # cell (0-based)
      9     idy=f['PM/idy'][:].astype(np.int64)-1   # peak (0-based)
     10     cnt=f['PM/count'][:]
     11     npeak=f['PM/peakStart'].shape[0]
---> 12     assert idx.max()<nbar and idy.m

In [ ]:
import numpy as np
for lbl,f in [('b1',f1),('b2',f2)]:
    idx=f['PM/idx'][:]; idy=f['PM/idy'][:]
    print(lbl,
          "| idx min/max:",idx.min(),idx.max(),
          "| idy min/max:",idy.min(),idy.max(),
          "| n peakStart:",f['PM/peakStart'].shape[0],
          "| n BD/name:",f['BD/name'].shape[0])
# how many idy values exceed peak array len?
idy=f1['PM/idy'][:]
npk=f1['PM/peakStart'].shape[0]
over=(idy>npk).sum()
print("\nb1 idy>npeak count:",over, "unique over vals:", np.unique(idy[idy>npk])[:20], "...n=",len(np.unique(idy[idy>npk])))


exit_code: 0
--- stdout ---
b1 | idx min/max: 1 10614 | idy min/max: 1 459975 | n peakStart: 459912 | n BD/name: 10614
b2 | idx min/max: 1 12374 | idy min/max: 1 459975 | n peakStart: 459912 | n BD/name: 12374

b1 idy>npeak count: 10265 unique over vals: [459913 459914 459915 459916 459917 459918 459919 459920 459921 459922
 459923 459924 459925 459926 459927 459928 459929 459930 459931 459932] ...n= 50

--- stderr ---



In [ ]:
import numpy as np
chrom=decode(np.array(f1['PM/peakChrom'])); st=f1['PM/peakStart'][:]; en=f1['PM/peakEnd'][:]
# unique chroms in order of appearance
seen=[]; 
for c in chrom:
    if not seen or seen[-1]!=c: seen.append(c)
print("chrom block order (peak coords):", seen)
print("\nfirst 3:", list(zip(chrom[:3],st[:3])))
print("last 5 peak coords:", list(zip(chrom[-5:],st[-5:],en[-5:])))
# HD/SQ contigs
sn=[x.decode() for x in np.array(f1['HD/SQ/SN'])]
print("\nHD/SQ contigs (66):", sn)
# which contigs are NOT represented in peak coords (candidates for missing 63)
print("\ncontigs in SQ but absent from peak coords:", [c for c in sn if c not in set(chrom)])


exit_code: 0
--- stdout ---
chrom block order (peak coords): ["b'chr1'", "b'chr10'", "b'chr11'", "b'chr12'", "b'chr13'", "b'chr14'", "b'chr15'", "b'chr16'", "b'chr17'", "b'chr18'", "b'chr19'", "b'chr1_GL456221_random'", "b'chr2'", "b'chr3'", "b'chr4'", "b'chr4_GL456216_random'", "b'chr4_JH584295_random'", "b'chr5'", "b'chr6'", "b'chr7'", "b'chr8'", "b'chr9'", "b'chrM'", "b'chrUn_GL456359'", "b'chrUn_GL456366'", "b'chrUn_GL456379'", "b'chrUn_GL456383'", "b'chrUn_GL456387'", "b'chrUn_GL456389'", "b'chrUn_GL456392'", "b'chrUn_GL456396'", "b'chrUn_JH584304'", "b'chrX'", "b'chrX_GL456233_random'", "b'chrY'"]

first 3: [("b'chr1'", 3012733), ("b'chr1'", 3094602), ("b'chr1'", 3094835)]
last 5 peak coords: [("b'chrY'", 90808562, 90808953), ("b'chrY'", 90808666, 90808940), ("b'chrY'", 90809423, 90809658), ("b'chrY'", 90810236, 90810402), ("b'chrY'", 90810803, 90811031)]

HD/SQ contigs (66): ['chr1', 'chr2', 'chrX', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr10', 'chr8', 'chr14', 'chr9', 'chr11

In [ ]:
import numpy as np, scipy.sparse as sp, pandas as pd, anndata as ad, h5py, gzip
NPK=459912

def dec(arr):
    out=[]
    for x in arr[:]:
        out.append(x.decode() if hasattr(x,'decode') else str(x))
    return np.array(out)

def build_biorep(f, label, sc_csv):
    bd_name=dec(f['BD/name']); nbar=len(bd_name)
    idx=f['PM/idx'][:].astype(np.int64)-1
    idy=f['PM/idy'][:].astype(np.int64)-1
    cnt=f['PM/count'][:]
    keep=idy<NPK
    dropped=(~keep).sum()
    idx,idy,cnt=idx[keep],idy[keep],cnt[keep]
    X=sp.coo_matrix((cnt,(idx,idy)), shape=(nbar,NPK)).tocsr()
    obs=pd.DataFrame(index=[f"{label}|{b}" for b in bd_name])
    for k in ['TN','UM','PP','UQ','SE','SA','PE','PL','US','CM']:
        if f'BD/{k}' in f: obs[k]=f[f'BD/{k}'][:]
    obs['barcode']=bd_name; obs['biorep']=label
    sc=sc_csv.drop_duplicates('barcode').set_index('barcode')
    cols=['passed_filters','peak_region_fragments','TSS_fragments','promoter_region_fragments',
          'enhancer_region_fragments','is__cell_barcode','cell_id']
    j=sc.reindex(bd_name)[cols]; j.index=obs.index
    obs=pd.concat([obs,j],axis=1)
    print(f"{label}: {X.shape} nnz={X.nnz} dropped_orphan_nnz={dropped}")
    return X,obs

X1,o1=build_biorep(f1,'biorep1',sc1)
X2,o2=build_biorep(f2,'biorep2',sc2)

chrom=dec(f1['PM/peakChrom']); st=f1['PM/peakStart'][:]; en=f1['PM/peakEnd'][:]
var=pd.DataFrame({'chr':chrom,'start':st,'end':en})
var.index=[f"{c}:{s}-{e}" for c,s,e in zip(chrom,st,en)]
print("var sample idx:", var.index[:2].tolist(), "| unique chroms:", var['chr'].nunique())

X=sp.vstack([X1,X2]).tocsr()
obs=pd.concat([o1,o2])
atac=ad.AnnData(X.astype(np.float32), obs=obs, var=var)
atac.uns['title']='Green et al 2023 (GSE232200) snATAC-seq of cortical Gad2+ inhibitory neurons (PPC) — peak x cell'
atac.uns['gsm']={'biorep1':'GSM7317753','biorep2':'GSM7317754'}
atac.uns['note']='Built from SnapATAC v1 .snap PM (peak) group, bin/genome mm10. 63 orphan peak indices without stored coords dropped.'
atac.write('/work/gse232200_snapatac.h5ad')
print("\nMERGED:",atac.shape,"nnz",atac.X.nnz)
print(atac.obs.groupby('biorep')[['passed_filters','peak_region_fragments','UQ']].median())
print("saved /work/gse232200_snapatac.h5ad")


exit_code: 137
--- stdout ---

--- stderr ---



In [ ]:
import subprocess
print(subprocess.run(['free','-g'],capture_output=True,text=True).stdout)
print(subprocess.run(['bash','-c','cat /sys/fs/cgroup/memory.max 2>/dev/null || cat /sys/fs/cgroup/memory/memory.limit_in_bytes 2>/dev/null'],capture_output=True,text=True).stdout)


exit_code: 1
--- stdout ---
(NOTE: the sandbox kernel was restarted before this cell — the previous container died (most likely OOM-killed). All in-memory variables, imports, and pip-installed packages from earlier cells are gone. You're now on a fresh kernel; re-load any data you need.)


--- stderr ---

FileNotFoundError: [Errno 2] No such file or directory: 'free'
---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 2
      1 import subprocess
----> 2 print(subprocess.run(['free','-g'],capture_output=True,text=True).stdout)
      3 print(subprocess.run(['bash','-c','cat /sys/fs/cgroup/memory.max 2>/dev/null || cat /sys/fs/cgroup/memory/memory.limit_in_bytes 2>/dev/null'],capture_output=True,text=True).stdout)

File /usr/local/lib/python3.11/subprocess.py:548, in run(input, capture_output, timeout, check, *popenargs, **kwargs)
    545     kwargs['stdout'] = PIPE
    546

In [ ]:
import os
def memgb():
    try:
        m=open('/sys/fs/cgroup/memory.max').read().strip()
        lim=float(m)/1e9 if m!='max' else None
    except: lim=None
    tot=os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1e9
    return lim, tot
print("cgroup limit GB, phys GB:", memgb())
import psutil if False else None


exit_code: 1
--- stdout ---

--- stderr ---

SyntaxError: invalid syntax (1856753048.py, line 10)
  Cell In[1], line 10
    import psutil if False else None
                  ^
SyntaxError: invalid syntax



In [ ]:
import os
m=open('/sys/fs/cgroup/memory.max').read().strip()
lim=float(m)/1e9 if m!='max' else 'unlimited'
print("cgroup mem limit GB:", lim)
cur=open('/sys/fs/cgroup/memory.current').read().strip()
print("current usage GB:", float(cur)/1e9)


exit_code: 0
--- stdout ---
cgroup mem limit GB: 8.589934592
current usage GB: 0.568549376

--- stderr ---



In [ ]:
import numpy as np, scipy.sparse as sp, pandas as pd, anndata as ad, h5py, gzip, gc
NPK=459912
base='/work/staged/green-2023-ppc-inhibitory-snatac'

def dec(arr):
    return np.array([x.decode() if hasattr(x,'decode') else str(x) for x in arr[:]])

def build_one(path, label, sc_path):
    f=h5py.File(path,'r')
    bd=dec(f['BD/name']); nbar=len(bd)
    idx=f['PM/idx'][:].astype(np.int32)-1
    idy=f['PM/idy'][:].astype(np.int32)-1
    cnt=f['PM/count'][:]                      # uint8
    keep=idy<NPK; ndrop=int((~keep).sum())
    idx=idx[keep]; idy=idy[keep]; cnt=cnt[keep]
    X=sp.csr_matrix((cnt.astype(np.float32),(idx,idy)), shape=(nbar,NPK))
    del idx,idy,cnt; gc.collect()
    obs=pd.DataFrame(index=[f"{label}|{b}" for b in bd])
    for k in ['TN','UM','PP','UQ','SE','SA','PE','PL','US','CM']:
        if f'BD/{k}' in f: obs[k]=f[f'BD/{k}'][:]
    obs['barcode']=bd; obs['biorep']=label
    sc=pd.read_csv(gzip.open(sc_path,'rt')).drop_duplicates('barcode').set_index('barcode')
    cols=['passed_filters','peak_region_fragments','TSS_fragments','promoter_region_fragments',
          'enhancer_region_fragments','is__cell_barcode','cell_id']
    j=sc.reindex(bd)[cols]; j.index=obs.index
    obs=pd.concat([obs,j],axis=1); del sc; gc.collect()
    # peaks (only from first file)
    var=None
    if label=='biorep1':
        var=pd.DataFrame({'chr':dec(f['PM/peakChrom']),'start':f['PM/peakStart'][:],'end':f['PM/peakEnd'][:]})
        var.index=[f"{c}:{s}-{e}" for c,s,e in zip(var['chr'],var['start'],var['end'])]
    f.close()
    print(f"{label}: X={X.shape} nnz={X.nnz} orphan_nnz_dropped={ndrop}")
    return X,obs,var

X1,o1,var=build_one(f'{base}/GSM7317753_biorep1.snap','biorep1',f'{base}/GSM7317753_biorep1_singlecell.csv.gz')
gc.collect()
X2,o2,_=build_one(f'{base}/GSM7317754_biorep2.snap','biorep2',f'{base}/GSM7317754_biorep2_singlecell.csv.gz')
gc.collect()
X=sp.vstack([X1,X2]).tocsr(); del X1,X2; gc.collect()
obs=pd.concat([o1,o2])
print("merged X:",X.shape,"nnz",X.nnz,"dtype",X.dtype)


exit_code: 1
--- stdout ---

--- stderr ---

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/work/staged/green-2023-ppc-inhibitory-snatac/GSM7317753_biorep1.snap', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)
---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[1], line 36
     32     f.close()
     33     print(f"{label}: X={X.shape} nnz={X.nnz} orphan_nnz_dropped={ndrop}")
     34     return X,obs,var
     35 
---> 36 X1,o1,var=build_one(f'{base}/GSM7317753_biorep1.snap','biorep1',f'{base}/GSM7317753_biorep1_singlecell.csv.gz')
     37 gc.collect()
     38 X2,o2,_=build_one(f'{base}/GSM7317754_biorep2.snap','biorep2',f'{base}/GSM7317754_biorep2_singlecell.csv.gz')
     39 gc.collect()

Cell In[1], line 9, in build_one(path, label, sc_path)
      8 def build_one(path, label, sc_path):
----> 9     f=h

In [ ]:
import os, glob
for p in glob.glob('/work/*.snap')+glob.glob('/work/staged/green-2023-ppc-inhibitory-snatac/*'):
    print(f"{os.path.getsize(p)/1e9:8.2f} GB  {p}")
print("---h5ads done so far---")
for p in glob.glob('/work/*.h5ad'):
    print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p}")


exit_code: 0
--- stdout ---
    5.57 GB  /work/GSM7317754_biorep2.snap
    5.48 GB  /work/GSM7317753_biorep1.snap
    0.01 GB  /work/staged/green-2023-ppc-inhibitory-snatac/GSM7317754_biorep2_singlecell.csv.gz
    2.26 GB  /work/staged/green-2023-ppc-inhibitory-snatac/GSM7317753_biorep1.snap.gz
    0.01 GB  /work/staged/green-2023-ppc-inhibitory-snatac/GSM7317753_biorep1_singlecell.csv.gz
    2.30 GB  /work/staged/green-2023-ppc-inhibitory-snatac/GSM7317754_biorep2.snap.gz
---h5ads done so far---
   519.1 MB  /work/gse136802_scrna.h5ad
    49.5 MB  /work/gse136802_atac.h5ad

--- stderr ---



In [ ]:
import numpy as np, scipy.sparse as sp, pandas as pd, anndata as ad, h5py, gzip, gc
NPK=459912
sbase='/work/staged/green-2023-ppc-inhibitory-snatac'

def dec(arr):
    return np.array([x.decode() if hasattr(x,'decode') else str(x) for x in arr[:]])

def build_one(snap, label, sc_path):
    f=h5py.File(snap,'r')
    bd=dec(f['BD/name']); nbar=len(bd)
    idx=f['PM/idx'][:].astype(np.int32)-1
    idy=f['PM/idy'][:].astype(np.int32)-1
    cnt=f['PM/count'][:]
    keep=idy<NPK; ndrop=int((~keep).sum())
    idx,idy,cnt=idx[keep],idy[keep],cnt[keep]
    X=sp.csr_matrix((cnt.astype(np.float32),(idx,idy)),shape=(nbar,NPK))
    del idx,idy,cnt; gc.collect()
    obs=pd.DataFrame(index=[f"{label}|{b}" for b in bd])
    for k in ['TN','UM','PP','UQ','SE','SA','PE','PL','US','CM']:
        if f'BD/{k}' in f: obs[k]=f[f'BD/{k}'][:]
    obs['barcode']=bd; obs['biorep']=label
    sc=pd.read_csv(gzip.open(sc_path,'rt')).drop_duplicates('barcode').set_index('barcode')
    cols=['passed_filters','peak_region_fragments','TSS_fragments','promoter_region_fragments',
          'enhancer_region_fragments','is__cell_barcode','cell_id']
    j=sc.reindex(bd)[cols]; j.index=obs.index
    obs=pd.concat([obs,j],axis=1); del sc; gc.collect()
    var=None
    if label=='biorep1':
        var=pd.DataFrame({'chr':dec(f['PM/peakChrom']),'start':f['PM/peakStart'][:],'end':f['PM/peakEnd'][:]})
        var.index=[f"{c}:{s}-{e}" for c,s,e in zip(var['chr'],var['start'],var['end'])]
    f.close()
    print(f"{label}: X={X.shape} nnz={X.nnz} orphan_dropped={ndrop}")
    return X,obs,var

X1,o1,var=build_one('/work/GSM7317753_biorep1.snap','biorep1',f'{sbase}/GSM7317753_biorep1_singlecell.csv.gz'); gc.collect()
X2,o2,_=build_one('/work/GSM7317754_biorep2.snap','biorep2',f'{sbase}/GSM7317754_biorep2_singlecell.csv.gz'); gc.collect()
X=sp.vstack([X1,X2]).tocsr(); del X1,X2; gc.collect()
obs=pd.concat([o1,o2])
print("MERGED:",X.shape,"nnz",X.nnz)


exit_code: 0
--- stdout ---
biorep1: X=(10614, 459912) nnz=120004793 orphan_dropped=10265
biorep2: X=(12374, 459912) nnz=122332676 orphan_dropped=8943
MERGED: (22988, 459912) nnz 242337469

--- stderr ---



In [ ]:
import anndata as ad, gc
atac=ad.AnnData(X, obs=obs, var=var)
atac.uns['title']='Green et al 2023 (GSE232200) snATAC-seq of cortical Gad2+ inhibitory neurons (mouse PPC) — peak x cell'
atac.uns['gsm']={'biorep1':'GSM7317753','biorep2':'GSM7317754'}
atac.uns['genome']='mm10'
atac.uns['source']='SnapATAC v1 .snap PM (peak) group; X = peak fragment counts (uint8 origin, stored float32).'
atac.uns['note']='63 orphan peak indices (idy 459913-459975) lacked stored coordinates and were dropped (~0.008% of nnz).'
print(atac)
print("\nQC medians by biorep:")
print(atac.obs.groupby('biorep')[['UQ','passed_filters','peak_region_fragments']].median())
print("\nis__cell_barcode counts:", atac.obs['is__cell_barcode'].value_counts(dropna=False).to_dict())
atac.write('/work/gse232200_snapatac.h5ad')
import os; print("\nsaved:", os.path.getsize('/work/gse232200_snapatac.h5ad')/1e9,"GB")
del X; gc.collect()


exit_code: 0
--- stdout ---
AnnData object with n_obs × n_vars = 22988 × 459912
    obs: 'TN', 'UM', 'PP', 'UQ', 'SE', 'SA', 'PE', 'PL', 'US', 'CM', 'barcode', 'biorep', 'passed_filters', 'peak_region_fragments', 'TSS_fragments', 'promoter_region_fragments', 'enhancer_region_fragments', 'is__cell_barcode', 'cell_id'
    var: 'chr', 'start', 'end'
    uns: 'title', 'gsm', 'genome', 'source', 'note'

QC medians by biorep:
              UQ  passed_filters  peak_region_fragments
biorep                                                 
biorep1  13201.5         13648.5                 6390.0
biorep2  12010.0         12294.5                 5970.0

is__cell_barcode counts: {1: 22876, 0: 112}

saved: 1.978446916 GB

[result] 0
--- stderr ---



In [ ]:
import anndata as ad, scanpy as sc
for p in ['/work/gse232200_snapatac.h5ad','/work/gse136802_scrna.h5ad','/work/gse136802_atac.h5ad']:
    a=ad.read_h5ad(p, backed='r')
    print(f"{p}\n   shape={a.shape} obs={list(a.obs.columns)[:6]}... layers={list(a.layers.keys())}")
    a.file.close()


exit_code: 0
--- stdout ---
/work/gse232200_snapatac.h5ad
   shape=(22988, 459912) obs=['TN', 'UM', 'PP', 'UQ', 'SE', 'SA']... layers=[]
/work/gse136802_scrna.h5ad
   shape=(47348, 27035) obs=['lane', 'gsm', 'assay']... layers=['counts', 'lognorm']
/work/gse136802_atac.h5ad
   shape=(6, 323369) obs=['celltype', 'replicate', 'region']... layers=['binary']

--- stderr ---



In [ ]:
import shutil, os, datetime
today=datetime.date.today().isoformat()
g232=f'/work/staged/green-2023-ppc-inhibitory-snatac'
g136=f'/work/staged/hrvatin-2019-pesca-interneuron'

# move configured h5ads into staged dirs
shutil.move('/work/gse232200_snapatac.h5ad', f'{g232}/gse232200_snapatac.h5ad')
shutil.move('/work/gse136802_scrna.h5ad', f'{g136}/gse136802_scrna.h5ad')
shutil.move('/work/gse136802_atac.h5ad', f'{g136}/gse136802_atac.h5ad')

cfg232=f"""# GSE232200 — provenance & schema

**Source:** NCBI GEO GSE232200 (Green et al. 2023, Nature, doi:10.1038/s41586-023-06357-1, PMID 37468637).
10x snATAC-seq of Gad2-Cre;Sun1-GFP cortical (PPC) inhibitory neurons. Downloaded {today} from
ftp.ncbi.nlm.nih.gov/geo/samples/GSM7317nnn/.

**Raw files committed:** GSM7317753_biorep1.snap.gz, GSM7317754_biorep2.snap.gz (SnapATAC v1, genome mm10),
plus per-biorep CellRanger-ATAC singlecell.csv.gz.

**Configured artifact:** gse232200_snapatac.h5ad — merged peak x cell matrix.
- shape: 22,988 cells x 459,912 peaks; X = peak fragment counts (float32, uint8 origin).
- obs: BD QC fields from .snap (TN,UM,PP,UQ,SE,SA,PE,PL,US,CM), barcode, biorep (biorep1=GSM7317753,
  biorep2=GSM7317754), plus joined CellRanger QC (passed_filters, peak_region_fragments, TSS_fragments,
  promoter/enhancer_region_fragments, is__cell_barcode, cell_id).
- var: chr/start/end (mm10); index = "chr:start-end".
- Median UQ ~12-13k fragments/cell; 22,876/22,988 flagged is__cell_barcode==1.

**Gotchas:**
- Built from the .snap `PM` (peak) group: idx=cell (1-based -> BD order), idy=peak (1-based -> peakChrom order).
- 63 orphan peak indices (idy 459913-459975) had NO stored coordinates; their entries (~0.008% of nnz)
  were dropped. Peak coordinate array is genomically sorted (chr1..chrY), so index<->coord is a valid prefix.
- Also available in .snap but NOT extracted: AM/5000 (5kb bin x cell matrix), FM (raw fragments), BD full QC.
"""
open(f'{g232}/gse232200_config.md','w').write(cfg232)

cfg136=f"""# GSE136802 — provenance & schema

**Source:** NCBI GEO GSE136802 (Hrvatin et al. 2019, eLife, doi:10.7554/eLife.48089). PESCA enhancer-AAV
platform: ATAC + scRNA of cortical Sst/Vip/Pv INTACT interneurons. Downloaded {today} from
ftp.ncbi.nlm.nih.gov/geo/series/GSE136nnn/GSE136802/.

**Raw files committed:** GSE136802_RAW.tar (10 scRNA counts.tsv.gz + 6 per-sample ATAC peak .bed.gz),
GSE136802_atac_master_counts.csv.gz, GSE136802_atac_master_binary_peak_table.csv.gz,
GSE136802_S2_annotated_gres.csv.gz.

**Configured artifact 1: gse136802_scrna.h5ad** (INTACT cortical interneuron scRNA-seq)
- shape: 47,348 cells x 27,035 genes. Concatenated from 10 lanes (GSM4058339_180320_6-19-*).
- X = raw counts; layers['counts']=raw; layers['lognorm']=log1p(CP10k).
- obs: lane, gsm, assay. Cell barcodes prefixed "<lane>|<barcode>".
- NOTE: cell-type (Sst/Vip/Pv) is assigned downstream by clustering — not provided as deposited metadata.

**Configured artifact 2: gse136802_atac.h5ad** (pseudobulk ATAC master peak matrix)
- shape: 6 samples x 323,369 peaks. obs = CTX_{{PV,VIP,SST}}_{{1,2}} (celltype, replicate, region).
- X = master_counts (fragment counts); layers['binary'] = binary peak-call table (same 6 samples).
- var: chr/start/end + conservation (MeanCons/MinCons/MaxCons) for all peaks; GRE annotation columns
  (Annotation, Gene_Name, Name, ATAC_Specificity, PESCA_Specificity) for the 287 annotated GREs
  (is_annotated_GRE flag). Peak index = "master_N".

**Gotchas:**
- ATAC data is pseudobulk (6 sorted samples), NOT single-cell.
- binary_peak_table has 12 cols (Chr/Start/End + 6 samples + 3 conservation); only the 6 sample cols form layers['binary'].
"""
open(f'{g136}/gse136802_config.md','w').write(cfg136)

# remove large decompressed .snap (keep .snap.gz as canonical raw)
for p in ['/work/GSM7317753_biorep1.snap','/work/GSM7317754_biorep2.snap']:
    os.remove(p)

print("== staged green ==")
for f in sorted(os.listdir(g232)): print(f"  {os.path.getsize(g232+'/'+f)/1e6:9.1f} MB  {f}")
print("== staged hrvatin ==")
for f in sorted(os.listdir(g136)): print(f"  {os.path.getsize(g136+'/'+f)/1e6:9.1f} MB  {f}")


exit_code: 0
--- stdout ---
== staged green ==
     2258.5 MB  GSM7317753_biorep1.snap.gz
        5.5 MB  GSM7317753_biorep1_singlecell.csv.gz
     2297.0 MB  GSM7317754_biorep2.snap.gz
        5.6 MB  GSM7317754_biorep2_singlecell.csv.gz
        0.0 MB  gse232200_config.md
     1978.4 MB  gse232200_snapatac.h5ad
== staged hrvatin ==
       51.5 MB  GSE136802_RAW.tar
        0.0 MB  GSE136802_S2_annotated_gres.csv.gz
        6.9 MB  GSE136802_atac_master_binary_peak_table.csv.gz
        2.2 MB  GSE136802_atac_master_counts.csv.gz
       49.5 MB  gse136802_atac.h5ad
        0.0 MB  gse136802_config.md
      519.1 MB  gse136802_scrna.h5ad

--- stderr ---



## Artifacts
- [`.mpl_cache/fontlist-v390.json`](./.mpl_cache/fontlist-v390.json)